In [1]:
import chess, chess.engine, os, stat
from policy import *
import random
from discrim import *
from file_helper import truncateGame

2026-04-23 09:07:18.611079: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-23 09:07:18.649909: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-23 09:07:19.716366: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/storage/icds/RISE/sw8/anaconda/conda_envs/pytorch/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version 

POLICY V8 (JOINT)


In [2]:
from stockfish import Stockfish
engine_path = r"./stockfish/src/stockfish"
sf = Stockfish(engine_path, parameters={"Threads": 1, "Hash": 256})
sf.set_depth(2)
sf.set_skill_level(2)
sf.get_engine_parameters()

{'Debug Log File': '',
 'Threads': 1,
 'Hash': 256,
 'Ponder': False,
 'MultiPV': 1,
 'Skill Level': 2,
 'Move Overhead': 10,
 'Slow Mover': 100,
 'UCI_Chess960': False,
 'UCI_LimitStrength': False,
 'UCI_Elo': 1350,
 'Contempt': 0,
 'Min Split Depth': 0,
 'Minimum Thinking Time': 20}

In [3]:
games= load_json("./data/Bijay_1549_games.json")
print(len(games))

Loading games: 100%|██████████| 392/392 [00:00<00:00, 589.89it/s]

392


In [4]:
import os

def simulate_games(agent, sf, num_games=400, file_dir="./data", file_postfix="0"):
    os.makedirs(file_dir, exist_ok=True)  # ✅ fix: create dir before writing
    
    games_data = []
    i = 0
    while i < num_games:
        board = chess.Board()
        moves = []
        aborted = False
        
        while not board.is_game_over():
            try:
                if board.turn == chess.WHITE:
                    move = agent.act(board)
                else:
                    sf.set_fen_position(board.fen())
                    best = sf.get_best_move()
                    if best is None:
                        aborted = True
                        break
                    move = chess.Move.from_uci(best)
            except Exception as e:
                print(f"[Game {i+1}] Error: {e}")
                aborted = True
                break
            
            board.push(move)
            moves.append(move.uci())
        
        if aborted:
            continue  # ✅ fix: skip broken games instead of saving them
        
        game_data = {
            "event": "Agent vs Stockfish",
            "round": i + 1,
            "white": f"Mimic Agent of {agent.id}",
            "black": "Stockfish",
            "result": board.result(),
            "moves": " ".join(moves)
        }
        if truncateGame(game_data):
            games_data.append(game_data)
            i += 1

    file_postfix = str(file_postfix).replace(".", "_")
    file_path = f"{file_dir}/{agent.id}_agent_vs_stockfish_{file_postfix}.json"
    with open(file_path, "w") as f:
        json.dump(games_data, f, indent=4)
    print(f"[Saved] {file_path}")  # ✅ confirms write succeeded
    return file_path

In [5]:
def overall_similarity_pipeline(json_A, json_B, player_A, player_B):

    print(f"\n--- FULL PIPELINE: {player_A} vs {player_B} ---\n")

    # ============================================================
    # 🔧 FIX: normalize JSON INSIDE PIPELINE (list → string)
    # ============================================================
    def normalize_json(path):
        with open(path, "r") as f:
            games = json.load(f)

        for g in games:
            if isinstance(g.get("moves"), list):
                g["moves"] = " ".join(g["moves"])

        return games

    # Write temporary cleaned versions (no external preprocessing step)
    import tempfile

    def write_temp(games):
        tmp = tempfile.NamedTemporaryFile(delete=False, mode="w", suffix=".json")
        json.dump(games, tmp)
        tmp.close()
        return tmp.name

    clean_A = write_temp(normalize_json(json_A))
    clean_B = write_temp(normalize_json(json_B))

    # ============================================================
    # ORIGINAL PIPELINE (UNCHANGED LOGIC BELOW)
    # ============================================================
    b_A, m_A, l_A = load_json_game_sequences(clean_A, player_A, 1.0)
    b_B, m_B, l_B = load_json_game_sequences(clean_B, player_B, 0.0)

    min_games = min(len(l_A), len(l_B))
    if min_games == 0:
        print("Not enough usable games.")
        return None

    b_A, m_A, l_A = b_A[:min_games], m_A[:min_games], l_A[:min_games]
    b_B, m_B, l_B = b_B[:min_games], m_B[:min_games], l_B[:min_games]

    raw_boards = b_A + b_B
    raw_moves  = m_A + m_B
    raw_labels = l_A + l_B

    combined = list(zip(raw_boards, raw_moves, raw_labels))
    random.shuffle(combined)
    raw_boards, raw_moves, raw_labels = zip(*combined)

    all_boards = np.array(raw_boards)
    all_moves  = np.array(raw_moves)
    all_labels = np.array(raw_labels)

    all_moves_onehot = tf.one_hot(all_moves, NUM_MOVES)

    model = build_style_classifier()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )

    model.fit(
        x={"board_seq": all_boards, "move_seq": all_moves_onehot},
        y=all_labels,
        batch_size=32,
        epochs=20,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    similarity = compute_overall_similarity(
        clean_A, clean_B, player_A, player_B, model
    )

    print(f"Overall playstyle similarity: {similarity:.2f}%")
    os.remove(clean_A)
    os.remove(clean_B)
    return similarity

In [6]:
def hyper_tuning(games,player_name,sf, num_games=400,file_dir="./data",player_file_dir="./data"):
    a_values = np.linspace(0, 1, 11)
    
    best_a = None
    best_score = float("-inf")
    agent = Agent(player_name,stockfish_path=r"./stockfish/src/stockfish")

    for a in a_values:
        try:

            agent.train(games,alpha=a)
            
            player_file_path = f"{player_file_dir}/{agent.id}_games.json"
            agent_file_path = simulate_games(
                agent,
                sf,
                num_games,
                file_dir=file_dir,
                file_postfix=f"_a_{a:.2f}"
            )
            
            score = overall_similarity_pipeline(
                player_file_path,
                agent_file_path,
                f"{agent.id}",
                f"Mimic Agent of {agent.id}"
            )

            print(f"a={a:.2f}, score={score:.3f}")

            if score > best_score:
                best_score = score
                best_a = a
        
        finally:
            # 🔥 CRITICAL: prevent Colab crashes
            import gc
            tf.keras.backend.clear_session()
            gc.collect()

    print(f"\nBest a: {best_a:.2f} (score={best_score:.3f})")

In [7]:
hyper_tuning(games,"Bijay_1549",sf,num_games=100,file_dir="./data100")

/storage/home/jmy5612/model V8/policy.py:123: UserWarning: Note that even though you've set Stockfish to play on a weaker elo or skill level, get_evaluation will still return full strength Stockfish's evaluation of the position.
  info = self.sf.get_evaluation()
/storage/home/jmy5612/model V8/policy.py:270: UserWarning: Note that even though you've set Stockfish to play on a weaker elo or skill level, get_top_moves will still return the top moves of full strength Stockfish.
  top = self.sf.get_top_moves(top_k)


Epoch 1/20
783/783 [==============================] - 303s 383ms/step - loss: 0.0025 - accuracy: 0.0958
Epoch 2/20
783/783 [==============================] - 300s 383ms/step - loss: 0.0019 - accuracy: 0.1099
Epoch 3/20
783/783 [==============================] - 300s 383ms/step - loss: 0.0017 - accuracy: 0.1108
Epoch 4/20
783/783 [==============================] - 300s 383ms/step - loss: 0.0015 - accuracy: 0.1127
Epoch 5/20
783/783 [==============================] - 300s 383ms/step - loss: 0.0013 - accuracy: 0.1139
Epoch 6/20
783/783 [==============================] - 300s 383ms/step - loss: 0.0011 - accuracy: 0.1145
Epoch 7/20
783/783 [==============================] - 300s 383ms/step - loss: 8.9649e-04 - accuracy: 0.1150
Epoch 8/20
783/783 [==============================] - 299s 382ms/step - loss: 7.4024e-04 - accuracy: 0.1146
Epoch 9/20
783/783 [==============================] - 300s 383ms/step - loss: 6.0234e-04 - accuracy: 0.1142
Epoch 10/20
783/783 [==============================]

KeyboardInterrupt: 

In [10]:
def play_sf_vs_sf(sf, num_games=400, file_dir="./data"):
    all_games = []

    for i in range(num_games):
        print(f"Game {i+1}/{num_games}")
        board = chess.Board()
        moves = []
        move_number = 1

        while not board.is_game_over():
            fen = board.fen()
            sf.set_fen_position(fen)
            move = sf.get_best_move()
            if move is None:
                break
            moves.append((board.turn, move_number, board.san(chess.Move.from_uci(move))))
            board.push(chess.Move.from_uci(move))
            if board.turn == chess.WHITE:
                move_number += 1

        pgn_moves = []
        for turn, num, san in moves:
            if turn == chess.WHITE:
                pgn_moves.append(f"{num}. {san}")
            else:
                pgn_moves.append(san)

        result = board.result()
        if result == "*":
            result = "1/2-1/2"

        all_games.append({
            "event": "SF vs SF",
            "white": "sf_white",
            "black": "Stockfish",
            "result": result,
            "moves": " ".join(pgn_moves)
        })

        print(f"Result: {result}")

    file_path = f"{file_dir}/sf_white_games.json"
    os.makedirs(file_dir, exist_ok=True)
    with open(file_path, "w") as f:
        json.dump(all_games, f, indent=4)
    print(f"Saved {num_games} games to {file_path}")

    return file_path

In [11]:
def get_sf_white_similarity(sf, player_name, file_dir="./data"):
    sf_json = play_sf_vs_sf(sf, file_dir=file_dir)
    player_json = f"{file_dir}/{player_name}_games.json"

    similarity = overall_similarity_pipeline(
        json_A=player_json,
        json_B=sf_json,
        player_A=player_name,
        player_B="sf_white"
    )

    print(f"Stockfish white vs {player_name} similarity: {similarity:.2f}%")
    return similarity

similarity = get_sf_white_similarity(
    sf,
    player_name="Bijay_1549"
)

Game 1/400


StockfishException: The Stockfish process has crashed

In [9]:
def evaluate_all_alphas(agent_id, player_file_dir="./data", file_dir="./data"):
    a_values = np.linspace(0, 1, 11)
    
    player_file_path = f"{player_file_dir}/{agent_id}_games.json"
    best_a = None
    best_score = float("-inf")

    for a in a_values:
        file_postfix = f"_a_{a:.2f}".replace(".", "_")
        agent_file_path = f"{file_dir}/{agent_id}_agent_vs_stockfish_{file_postfix}.json"
        
        if not os.path.exists(agent_file_path):
            print(f"a={a:.2f}, file not found: {agent_file_path}")
            continue
        
        try:
            score = overall_similarity_pipeline(
                player_file_path,
                agent_file_path,
                f"{agent_id}",
                f"Mimic Agent of {agent_id}"
            )
            print(f"a={a:.2f}, score={score:.3f}")
            
            if score > best_score:
                best_score = score
                best_a = a
        except Exception as e:
            print(f"a={a:.2f}, error: {e}")

    if best_a is not None:
        print(f"\nBest a: {best_a:.2f} (score={best_score:.3f})")
    else:
        print("\nNo successful evaluations.")
    
    return best_a, best_score
evaluate_all_alphas("Bijay_1549",player_file_dir="./data",file_dir="./data100")


--- FULL PIPELINE: Bijay_1549 vs Mimic Agent of Bijay_1549 ---

Epoch 1/20
5/5 [==============================] - 16s 2s/step - loss: 1.8111 - accuracy: 0.6218 - val_loss: 1.7970 - val_accuracy: 0.6500
Epoch 2/20
5/5 [==============================] - 8s 2s/step - loss: 1.7867 - accuracy: 0.6987 - val_loss: 1.7723 - val_accuracy: 0.6000
Epoch 3/20
5/5 [==============================] - 9s 2s/step - loss: 1.7590 - accuracy: 0.7308 - val_loss: 1.7469 - val_accuracy: 0.6000
Epoch 4/20
5/5 [==============================] - 8s 2s/step - loss: 1.7313 - accuracy: 0.7308 - val_loss: 1.7156 - val_accuracy: 0.6750
Epoch 5/20
5/5 [==============================] - 8s 2s/step - loss: 1.6920 - accuracy: 0.8141 - val_loss: 1.6759 - val_accuracy: 0.7750
Epoch 6/20
5/5 [==============================] - 8s 2s/step - loss: 1.6434 - accuracy: 0.8526 - val_loss: 1.6143 - val_accuracy: 0.8250
Epoch 7/20
5/5 [==============================] - 8s 2s/step - loss: 1.5556 - accuracy: 0.9167 - val_loss: 1.503

(0.9, 16.541465835686974)